In [5]:
import pandas as pd
from sklearn.preprocessing import RobustScaler

print("Starting Epic 3, Story 1: Loading Datasets...")

# Ensure the new pair of datasets are loaded
if 'df' not in globals():
    try:
        # ⚠️ PASTE YOUR COPIED FILE PATHS HERE ⚠️
        # Make sure the 'r' stays outside the quotes to handle Windows backslashes
        app_path = '../creditcard_csv/application_record.csv'
        credit_path = '../creditcard_csv/credit_record.csv'
        
        # Load datasets
        app_df = pd.read_csv(app_path)
        print(f"Loaded Application Features Successfully! Shape: {app_df.shape}")
        
        credit_df = pd.read_csv(credit_path)
        print(f"Loaded Credit History Successfully! Shape: {credit_df.shape}\n")
        
        # Simulating the merged 'df' so the notebook doesn't break prematurely
        df = app_df.copy() 
        
    except FileNotFoundError as fnf_error:
        print(f"\n❌ ERROR: Python cannot find the file at this exact location:\n{fnf_error.filename}")
        print("Please check File Explorer and update the 'app_path' or 'credit_path' variables.")
        raise
    except Exception as e:
        raise NameError(f"An unexpected error occurred: {e}")

print(f"Current Working Dataset (df) Shape: {df.shape}\n")
print("-" * 40)

Starting Epic 3, Story 1: Loading Datasets...
Current Working Dataset (df) Shape: (438557, 18)

----------------------------------------


In [6]:
print("1. Dropping Duplicates...")
# The SmartBridge dataset often contains duplicated applicant entries (e.g., multiple applications from the same person).
# If the 'ID' column is still present, it will mask duplicates, so we check all columns EXCEPT 'ID'.

if 'ID' in df.columns:
    # Check for duplicates ignoring the unique ID column
    subset_cols = [col for col in df.columns if col != 'ID']
    duplicates_count = df.duplicated(subset=subset_cols).sum()
else:
    # If ID is already dropped, do a standard full-row check
    duplicates_count = df.duplicated().sum()

print(f"Found {duplicates_count} duplicate rows.")

if duplicates_count > 0:
    if 'ID' in df.columns:
        df.drop_duplicates(subset=subset_cols, keep='first', inplace=True)
    else:
        df.drop_duplicates(keep='first', inplace=True)
        
    print(f"Duplicates removed. New Shape: {df.shape}")
else:
    print("No duplicates found.")
    
print("-" * 40)

1. Dropping Duplicates...
Found 348472 duplicate rows.
Duplicates removed. New Shape: (90085, 18)
----------------------------------------


In [7]:
print("2. Handling Missing Values...")

# It is much more helpful to see exactly WHICH columns are missing data
missing_total = df.isnull().sum().sum()
print(f"Total missing values found: {missing_total}")

if missing_total > 0:
    missing_cols = df.columns[df.isnull().any()]
    print(f"\nBreakdown of missing values:\n{df[missing_cols].isnull().sum()}\n")

    # 1. Handle Categorical Missing Values (Crucial for SmartBridge 'OCCUPATION_TYPE')
    categorical_cols = df.select_dtypes(include=['object']).columns
    for col in categorical_cols:
        if df[col].isnull().sum() > 0:
            df[col] = df[col].fillna('Unknown')
            print(f"-> Filled missing values in categorical column '{col}' with 'Unknown'.")

    # 2. Handle Numerical Missing Values (Your original median strategy as a safe fallback)
    numeric_cols = df.select_dtypes(include=['number']).columns
    for col in numeric_cols:
        if df[col].isnull().sum() > 0:
            df[col] = df[col].fillna(df[col].median())
            print(f"-> Filled missing values in numerical column '{col}' with median.")
            
    print("\nAll missing values have been successfully handled.")
else:
    print("No missing values to handle.")
    
print("-" * 40)

2. Handling Missing Values...
Total missing values found: 27477

Breakdown of missing values:
OCCUPATION_TYPE    27477
dtype: int64

-> Filled missing values in categorical column 'OCCUPATION_TYPE' with 'Unknown'.

All missing values have been successfully handled.
----------------------------------------


In [8]:
print("3. Data Cleaning, Aggregating, and Merging...")

# 1. Clean column names (always a good practice)
app_df.columns = app_df.columns.str.strip()
credit_df.columns = credit_df.columns.str.strip()
print("Column names stripped of leading/trailing spaces.")

# 2. Aggregate Credit History to create the Target Variable
# STATUS definitions: 'C'=paid, 'X'=no loan, '0'=1-29 days, '1'=30-59 days
# '2'=60-89 days, '3'=90-119, '4'=120-149, '5'=>150 days
# You defined High Risk as 60+ days past due (Status 2, 3, 4, or 5).
high_risk_statuses = ['2', '3', '4', '5']

# Create a binary column per month: 1 if high risk, 0 otherwise
credit_df['Risk_Flag'] = credit_df['STATUS'].isin(high_risk_statuses).astype(int)

# Group by Applicant ID. If the max is 1, they were 60+ days late AT LEAST once.
credit_agg = credit_df.groupby('ID')['Risk_Flag'].max().reset_index()
credit_agg.rename(columns={'Risk_Flag': 'Is_High_Risk'}, inplace=True)
print("Credit history aggregated. Target variable 'Is_High_Risk' created.")

# 3. Merge the datasets
# Inner join ensures we only keep applicants who have both features and a credit history
df = pd.merge(app_df, credit_agg, on='ID', how='inner')
print(f"Datasets successfully inner-joined. Merged Shape: {df.shape}")

# 4. Drop the ID column
# IDs are unique identifiers that add noise and can cause overfitting in ML models
df.drop('ID', axis=1, inplace=True)
print("Dropped 'ID' column to prepare for Machine Learning.")

print("-" * 40)

3. Data Cleaning, Aggregating, and Merging...
Column names stripped of leading/trailing spaces.
Credit history aggregated. Target variable 'Is_High_Risk' created.
Datasets successfully inner-joined. Merged Shape: (36457, 19)
Dropped 'ID' column to prepare for Machine Learning.
----------------------------------------


In [9]:
print("4. Feature Engineering...")
# The SmartBridge dataset doesn't have 'Time' or 'Amount', but it has 'AMT_INCOME_TOTAL', 'DAYS_BIRTH', and 'DAYS_EMPLOYED'.

# 1. Convert negative days into readable positive years
if 'DAYS_BIRTH' in df.columns:
    df['AGE_YEARS'] = round(df['DAYS_BIRTH'] / -365.25, 1)
    df.drop('DAYS_BIRTH', axis=1, inplace=True)
    print("-> Converted 'DAYS_BIRTH' to 'AGE_YEARS'.")

if 'DAYS_EMPLOYED' in df.columns:
    # In this dataset, positive values (usually 365243) mean unemployed/pensioner. We map those to 0 years.
    df['YEARS_EMPLOYED'] = df['DAYS_EMPLOYED'].apply(lambda x: 0 if x > 0 else abs(x) / 365.25)
    df['YEARS_EMPLOYED'] = round(df['YEARS_EMPLOYED'], 1)
    df.drop('DAYS_EMPLOYED', axis=1, inplace=True)
    print("-> Converted 'DAYS_EMPLOYED' to 'YEARS_EMPLOYED' (and handled unemployed outliers).")

# 2. Scale Income using RobustScaler (Excellent for handling extreme high-income outliers)
if 'AMT_INCOME_TOTAL' in df.columns:
    # Ensure scaler is defined (using RobustScaler from your earlier imports)
    scaler = RobustScaler()
    
    df['Scaled_Income'] = scaler.fit_transform(df['AMT_INCOME_TOTAL'].values.reshape(-1, 1))
    df.drop(['AMT_INCOME_TOTAL'], axis=1, inplace=True)
    
    # Optional: Reorder columns to move the scaled feature to the front
    scaled_income = df.pop('Scaled_Income')
    df.insert(0, 'Scaled_Income', scaled_income)
    
    print("-> 'AMT_INCOME_TOTAL' has been scaled using RobustScaler.")
else:
    print("-> 'AMT_INCOME_TOTAL' not found. Skipping scaling.")

print("-" * 40)

4. Feature Engineering...
-> Converted 'DAYS_BIRTH' to 'AGE_YEARS'.
-> Converted 'DAYS_EMPLOYED' to 'YEARS_EMPLOYED' (and handled unemployed outliers).
-> 'AMT_INCOME_TOTAL' has been scaled using RobustScaler.
----------------------------------------


In [10]:
print("5. Handling Categorical Values...")
# Dynamically grab any remaining text/categorical columns
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()

# The new target variable is 'Is_High_Risk' (just in case it was accidentally cast as an object)
if 'Is_High_Risk' in categorical_cols:
    categorical_cols.remove('Is_High_Risk') 

if len(categorical_cols) > 0:
    print(f"Encoding columns: {categorical_cols}")
    
    # CRITICAL: Added dtype=int to ensure we get 0/1 integers instead of True/False booleans
    df = pd.get_dummies(df, columns=categorical_cols, drop_first=True, dtype=int)
    
    print("Categorical variables successfully One-Hot Encoded.")
else:
    print("No categorical columns found. Dataset is strictly numerical.")
    
print("-" * 40)

print(f"\n🚀 Pre-processing Complete! Final Dataset Shape: {df.shape}")

5. Handling Categorical Values...
Encoding columns: ['CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE']
Categorical variables successfully One-Hot Encoded.
----------------------------------------

🚀 Pre-processing Complete! Final Dataset Shape: (36457, 47)
